# Week 6: RAG Pipeline with Retrieval Evaluation
**Dalien Cable | COSC 650 Applied LLM Systems**

This notebook extends the Week 5 RAG pipeline over 8 diabetes care and prevention documents. I kept the Week 5 setup (PDF cleaning, by-sentence chunks, all-MiniLM-L6-v2 in FAISS, the same 10 questions) and tested three changes: two ways of rewriting the question before searching (pseudo-relevance feedback and HyDE) and a cross-encoder re-ranker. Each one is measured against plain search, and Part 3 explains a case where a change made things worse.

### Corpus
All documents are PDFs, because that is how the majority of scientific papers and clinical guidelines are published. The PDFs are kept locally and are not committed to this repository to ensure no copyright issues or redistribution to 3rd party sites like GitHub occur.

| File | Document | Publisher |
|---|---|---|
| cdc_174262_DS1 | Newer Pharmacologic Treatments in Adults With Type 2 Diabetes (clinical guideline, author manuscript) | American College of Physicians, via CDC Stacks |
| dprp-standards | Diabetes Prevention Recognition Program Standards | CDC |
| dsa509 | MANAGEMENT OF DIABETES MELLITUS STANDARDS OF CARE AND CLINICAL PRACTICE GUIDELINES | WHO Regional Office for the Eastern Mediterranean |
| guidance-on-global-monitoring-for-diabetes | Guidance on global monitoring for diabetes prevention and control | WHO |
| HPDP_Diabetes_guide_for_diabetes_care | Diabetes Care Guidelines | Vermont Department of Health |
| On-your-way-to-preventing-type-2-diabetes | On Your Way to Preventing Type 2 Diabetes | CDC |
| Optimal_Diabetes_Care | Optimal Diabetes Care specifications | MN Community Measurement |
| who-ucn-ncd-20.1-eng | HEARTS-D: Diagnosis and management of type 2 diabetes | WHO |

## Setup
Loads the embedding model. `HF_HOME` stores the downloaded model in a local `.hf_cache` folder, which is excluded from the repo in `.gitignore`.

In [1]:
import os, re, pathlib, numpy as np
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer
import faiss
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Foundation: Week 5 pipeline
### Load the PDFs
Reads each PDF with pypdf, joins its pages into one text, and records the page count. The page count is needed by the cleaning step below.

In [2]:
from pypdf import PdfReader

CORPUS = pathlib.Path('corpus')
DOCS = {}    # file name -> the document's text
PAGES = {}   # file name -> how many pages it has

for path in sorted(CORPUS.glob('*.pdf')):
    reader = PdfReader(path)
    text = '\n'.join(page.extract_text() or '' for page in reader.pages)
    DOCS[path.stem] = text
    PAGES[path.stem] = len(reader.pages)
    print(f'{path.stem}: {len(reader.pages)} pages, {len(text):,} chars')

cdc_174262_DS1: 19 pages, 53,097 chars
dprp-standards: 65 pages, 166,731 chars
dsa509: 35 pages, 51,807 chars
guidance-on-global-monitoring-for-diabetes: 94 pages, 215,230 chars
HPDP_Diabetes_guide_for_diabetes_care: 2 pages, 4,197 chars
On-your-way-to-preventing-type-2-diabetes: 15 pages, 25,120 chars
Optimal_Diabetes_Care: 4 pages, 7,917 chars
who-ucn-ncd-20.1-eng: 35 pages, 62,460 chars


### Clean the extracted text
PDF extraction pulls out everything printed on a page, including running headers, footers, and page numbers. Left in, these lines get copied into many chunks and make unrelated chunks look alike to the retriever. I first saw this in an earlier exercise, where an FDA guidance document's running header ended up inside chunks, and the fix is to remove such lines at extraction time, before chunking, while each one is still a single intact line. Rather than hard-coding one header, this step detects any line that repeats across pages, so it works on all 8 documents without per-document rules.

**Rule.** A line is removed if (1) it appears on at least 40% of a document's pages (documents of 3+ pages only), or (2) it contains no letters, such as page numbers, stray dashes, and number-only fields from fill-in templates.

**Why 40%.** Some headers alternate between left and right pages, so they appear on about half the pages. 40% catches them but keeps repeated content, such as "Method of estimation" in the WHO monitoring guidance (32% of pages). Documents under 3 pages skip test 1, because on a 2-page document any line appearing twice would count.

**Checks.** Before settling the rule I listed the most repeated lines per document and checked borderline cases in the PDFs: "Insulin" in dsa509 is a running header, and the repeated number-colon-number lines in the prevention booklet belong to its fill-in templates, so removing both is correct. Each document shrank by less than 10% (output below), so the rule removes noise.

**Not handled.** Reference lists, author lists, and table text are left in.

In [3]:
from collections import Counter

def line_key(line):
    return re.sub(r'\d+', '#', line.strip())

def clean(text, n_pages):
    counts = Counter(line_key(l) for l in text.splitlines() if l.strip())
    kept = []
    for line in text.splitlines():
        s = line.strip()
        repeats_a_lot = n_pages >= 3 and counts[line_key(line)] >= 0.4 * n_pages
        only_numbers_or_symbols = s != '' and re.fullmatch(r'[\d\W_]+', s) is not None
        if not (repeats_a_lot or only_numbers_or_symbols):
            kept.append(line)
    return '\n'.join(kept)

CLEAN = {}
for name, text in DOCS.items():
    CLEAN[name] = clean(text, PAGES[name])
    print(f'{name}: {len(text):,} -> {len(CLEAN[name]):,} chars')

cdc_174262_DS1: 53,097 -> 49,981 chars
dprp-standards: 166,731 -> 162,671 chars
dsa509: 51,807 -> 51,089 chars
guidance-on-global-monitoring-for-diabetes: 215,230 -> 210,896 chars
HPDP_Diabetes_guide_for_diabetes_care: 4,197 -> 4,197 chars
On-your-way-to-preventing-type-2-diabetes: 25,120 -> 24,996 chars
Optimal_Diabetes_Care: 7,917 -> 7,035 chars
who-ucn-ncd-20.1-eng: 62,460 -> 60,839 chars


### Chunking and search
I'm only using by-sentence chunks this week. They're whole sentences packed into pieces of up to 500 characters, cut separately for each document so nothing mixes two sources. When I checked the rankings in the discussion work, this setup found the answer for all five questions I tested, just not always near the top. That matters because a re-ranker can only reorder what the search already found. Search is all-MiniLM-L6-v2 with FAISS.

In [4]:
def chunk_sentences(text, max_chars=500):
    text = re.sub(r'\s+', ' ', text).strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current = [], ''
    for s in sentences:
        if current and len(current) + len(s) + 1 > max_chars:
            chunks.append(current)
            current = s
        else:
            current = (current + ' ' + s).strip()
    if current:
        chunks.append(current)
    return chunks

def chunk_corpus(chunker):
    chunks, sources = [], []
    for name, text in CLEAN.items():
        for c in chunker(text):
            chunks.append(c)
            sources.append(name)
    return chunks, sources

def build(chunks):
    e = embedder.encode(chunks, normalize_embeddings=True).astype('float32')
    idx = faiss.IndexFlatIP(e.shape[1]); idx.add(e); return idx
def retrieve(idx, chunks, q, k=3):
    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')
    # FAISS uses -1 for missing results when fewer than k chunks exist.
    return [chunks[i] for i in idx.search(qe, k)[1][0] if i >= 0]

# Week 6 uses one chunking config. The dicts keep the Week 5 names so later cells work unchanged.
configs, SOURCES = {}, {}
configs['by-sentence'], SOURCES['by-sentence'] = chunk_corpus(chunk_sentences)
indices = {'by-sentence': build(configs['by-sentence'])}
print(f"by-sentence -> {len(configs['by-sentence'])} chunks")

by-sentence -> 1209 chunks


### The 10 questions
These are the same 10 questions as Week 5, each paired with a short fact copied from the documents. I wrote the facts before running any search, and they decide what counts as relevant. Some questions use the documents' own wording and some are plain-language versions, like "if sugar stays high". Two ask about telling type 1 from type 2, where the passages share almost all their words.

In [5]:
qa = [
    ('what A1C level is used to diagnose diabetes',                       '6.5%'),
    ('what fasting blood sugar confirms a diabetes diagnosis',            '126 mg/dl'),
    ('which medicines should be added to metformin if sugar stays high',  'SGLT-2 inhibitor or GLP-1 agonist'),
    ('how long does a lifestyle change program last',                     '12 months'),
    ('how many sessions are in the first six months of the program',       '16 sessions'),
    ('how much weight should someone lose to prevent type 2 diabetes',     '5% to 7%'),
    ('how much exercise per week is recommended',                         '150 minutes'),
    ('what blood pressure target counts as optimal diabetes care',         '<140'),
    ('what signs suggest type 1 rather than type 2 diabetes',              'ketones'),
    ('when should a newly diagnosed person be treated as type 1',          'insulin'),
]

### Checking if facts exist
Before trusting the scores I checked that each fact appears in at least one of the cleaned documents. If one was missing, its question could not be answered correctly. I changed the blood pressure fact from 140/90 to <140 so it matches the wording in the source.

In [6]:
missing = [fact for q, fact in qa
           if not any(fact.lower() in re.sub(r'\s+', ' ', t).lower() for t in CLEAN.values())]
print('facts not found in any document:', missing if missing else 'none')

facts not found in any document: none


### Gemini helper
A small function that sends a prompt to Gemini and returns the reply. I use gemini-3.5-flash & HyDE uses it below. It returns nothing if no key is found.

In [7]:
from dotenv import load_dotenv
load_dotenv('.env')

def gemini_chat(messages, model='gemini-3.5-flash', **kw):
    if not os.environ.get('GEMINI_API_KEY'):
        return None
    from openai import OpenAI
    client = OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message.content

## Part 1: Transform and re-rank

Everything above is the Week 5 pipeline. From here I tried three changes and measured each one against the plain search below: pseudo-relevance feedback (PRF) and HyDE, which both rewrite the question before searching, and a cross-encoder re-ranker, which reorders the results after searching.

### Starting point: plain search
Each question goes into the search as written.  Two numbers per question: how many of the top 3 chunks contain the answer, and where the first chunk with the answer lands in the top 100. Everything after this gets compared to this table.

In [8]:
WK_CONFIG = 'by-sentence'
WK_CHUNKS = configs[WK_CONFIG]
WK_K = 3       
WK_DEPTH = 100

def search_baseline(q):
    return retrieve(indices[WK_CONFIG], WK_CHUNKS, q, k=WK_DEPTH)

def measure(search):
    """Run one search method on all 10 questions.
    Returns (precision@3, rank of first chunk containing the fact) per question."""
    out = []
    for q, fact in qa:
        ranked = search(q)
        hits = sum(1 for ch in ranked[:WK_K] if fact.lower() in ch.lower())
        rank = next((n for n, ch in enumerate(ranked, start=1)
                     if fact.lower() in ch.lower()), None)
        out.append((hits / WK_K, rank))
    return out

wk_baseline = measure(search_baseline)

print(f"{'question':58s} {'P@3':>5s} {'first rank':>11s}")
for (q, fact), (p, rank) in zip(qa, wk_baseline):
    print(f'{q[:58]:58s} {p:5.2f} {str(rank) if rank else ">100":>11s}')
print(f"\nmean precision@3: {sum(p for p, _ in wk_baseline) / len(wk_baseline):.2f}")

question                                                     P@3  first rank
what A1C level is used to diagnose diabetes                 0.33           1
what fasting blood sugar confirms a diabetes diagnosis      0.00           5
which medicines should be added to metformin if sugar stay  0.00          30
how long does a lifestyle change program last               0.00          52
how many sessions are in the first six months of the progr  0.00          14
how much weight should someone lose to prevent type 2 diab  0.00        >100
how much exercise per week is recommended                   1.00           1
what blood pressure target counts as optimal diabetes care  0.00          35
what signs suggest type 1 rather than type 2 diabetes       0.00          27
when should a newly diagnosed person be treated as type 1   0.00           4

mean precision@3: 0.13


### Comparing two runs
A small function that lines up two sets of results question by question. A question counts as improved if the first chunk with the answer moved higher up the list, and regressed if it dropped lower. Precision@3 is shown alongside.

In [9]:
def compare(before, after):
    better = worse = 0
    print(f"{'question':44s} {'P@3':>11s}   {'first rank':>12s}   verdict")
    for (q, _), (pb, rb), (pa, ra) in zip(qa, before, after):
        rb_n, ra_n = rb or 101, ra or 101   # no answer in the top 100 counts as rank 101
        verdict = 'IMPROVED' if ra_n < rb_n else 'REGRESSED' if ra_n > rb_n else 'same'
        better += verdict == 'IMPROVED'
        worse += verdict == 'REGRESSED'
        rb_s = str(rb) if rb else '>100'
        ra_s = str(ra) if ra else '>100'
        print(f'{q[:44]:44s} {pb:.2f} -> {pa:.2f}   {rb_s:>5s} -> {ra_s:<5s}   {verdict}')
    mean_b = sum(p for p, _ in before) / len(before)
    mean_a = sum(p for p, _ in after) / len(after)
    print(f'\nimproved {better}, regressed {worse}, same {len(qa) - better - worse}')
    print(f'mean precision@3: {mean_b:.2f} -> {mean_a:.2f}')

In [10]:
import re
from collections import Counter

WK_STOP = set('a an the of to in on for and or is are how do does what so that with by as it why'.split())

def prf_expand(q, feedback_k=3, add=4):
    top = retrieve(indices[WK_CONFIG], WK_CHUNKS, q, k=feedback_k)
    words = re.findall(r'[a-zA-Z]+', ' '.join(top).lower())
    cand = [w for w in words if w not in WK_STOP and len(w) > 3 and w not in q.lower()]
    return q + ' ' + ' '.join(w for w, _ in Counter(cand).most_common(add))

def search_prf(q):
    return search_baseline(prf_expand(q))

### PRF: helpers

The idea is to let the search help rewrite the question. `prf_expand` runs the question once, takes the most common words from the top 3 chunks (skipping short words, stop words and words already in the question), and adds the top 4 to the end. `search_prf` then searches again with that longer question. It only works if that first top 3 was any good.

### Running PRF

Same 10 questions, compared against the baseline.

In [11]:
wk_prf = measure(search_prf)
compare(wk_baseline, wk_prf)

question                                             P@3     first rank   verdict
what A1C level is used to diagnose diabetes  0.33 -> 0.33       1 -> 1       same
what fasting blood sugar confirms a diabetes 0.00 -> 0.00       5 -> 6       REGRESSED
which medicines should be added to metformin 0.00 -> 0.00      30 -> 29      IMPROVED
how long does a lifestyle change program las 0.00 -> 0.00      52 -> 77      REGRESSED
how many sessions are in the first six month 0.00 -> 0.00      14 -> 21      REGRESSED
how much weight should someone lose to preve 0.00 -> 0.00    >100 -> >100    same
how much exercise per week is recommended    1.00 -> 1.00       1 -> 1       same
what blood pressure target counts as optimal 0.00 -> 0.00      35 -> 9       IMPROVED
what signs suggest type 1 rather than type 2 0.00 -> 0.00      27 -> 31      REGRESSED
when should a newly diagnosed person be trea 0.00 -> 0.33       4 -> 2       IMPROVED

improved 3, regressed 4, same 3
mean precision@3: 0.13 -> 0.17


### What PRF added

Before judging PRF, I printed the extra words it added to each question. This shows whether the top 3 chunks gave it useful vocabulary or just noise.

In [12]:
for q, _ in qa:
    print(f'{q[:50]:50s} + {prf_expand(q)[len(q):].strip()}')

what A1C level is used to diagnose diabetes        + prediabetes your type american
what fasting blood sugar confirms a diabetes diagn + glucose should values ongoing
which medicines should be added to metformin if su + evidence treatment term contraindicated
how long does a lifestyle change program last      + least this coaches training
how many sessions are in the first six months of t + retention beginning eligible requirement
how much weight should someone lose to prevent typ + your risk lower advised
how much exercise per week is recommended          + activity physical least should
what blood pressure target counts as optimal diabe + people control mmhg guidelines
what signs suggest type 1 rather than type 2 diabe + most common insulin overweight
when should a newly diagnosed person be treated as + diabetes diagnosis present this


### Checking what came back

`peek` prints the top 3 chunks for one question and marks whether each one contains the fact. I use it here on the lifestyle program question to see what PRF did to the results, and again later for the failure case.

In [13]:
def peek(question_number, search, n=3):
    q, fact = qa[question_number]
    print(f'{q}   | fact: {fact}')
    for ch in search(q)[:n]:
        src = SOURCES[WK_CONFIG][WK_CHUNKS.index(ch)]
        mark = 'RELEVANT' if fact.lower() in ch.lower() else 'miss    '
        print(f'  [{mark}] ({src}) {ch[:150]}')

print('BASELINE'); peek(3, search_baseline)
print('\nPRF');      peek(3, search_prf)

BASELINE
how long does a lifestyle change program last   | fact: 12 months
  [miss    ] (On-your-way-to-preventing-type-2-diabetes) This guide can’t substitute for this lifestyle change program. But if you can’t join one right now, or if you want a jumpstart before your program beg
  [miss    ] (dprp-standards) All Lifestyle Coaches and Program Coordinators should receive additional training each time CDC revises the DPRP Standards, and CDC will offer such tr
  [miss    ] (dprp-standards) Nat ional DPP lifestyle change program (National DPP LCP) = The translated adaptation of the DPP lifestyle intervention which: • Is a yearlong, struct

PRF
how long does a lifestyle change program last   | fact: 12 months
  [miss    ] (dprp-standards) All Lifestyle Coaches and Program Coordinators should receive additional training each time CDC revises the DPRP Standards, and CDC will offer such tr
  [miss    ] (dprp-standards) Training Recognized organizations are responsible for ensuring that an ad

### HyDE: helpers

HyDE stands for hypothetical document embeddings. Instead of searching with the question, I asked Gemini to write a short guideline-style answer first and search with that. The made-up answer uses clinical wording, so technically it should be closer to the real chunks vs a plain-language question. Each answer is saved in `HYDE_CACHE`, this means rerunning a cell doesn't call Gemini again. I set the temperature to 0 to try and ensure a more consistent output.

In [14]:
HYDE_CACHE = {}

def hyde_text(q):
    if q not in HYDE_CACHE:
        prompt = ('Write a short passage (2 to 3 sentences) in the style of a clinical '
                  'diabetes guideline that answers this question. Use precise clinical '
                  'terminology. Question: ' + q)
        out = gemini_chat([{'role': 'user', 'content': prompt}], temperature=0)
        if not out:
            raise RuntimeError('No Gemini key found: check .env is in this folder')
        HYDE_CACHE[q] = out
    return HYDE_CACHE[q]

def search_hyde(q):
    return search_baseline(hyde_text(q))

### Run HyDE

Same comparison against the baseline. This cell calls Gemini, once per question, but it can take a while.

In [15]:
wk_hyde = measure(search_hyde)
compare(wk_baseline, wk_hyde)

question                                             P@3     first rank   verdict
what A1C level is used to diagnose diabetes  0.33 -> 0.33       1 -> 1       same
what fasting blood sugar confirms a diabetes 0.00 -> 0.33       5 -> 3       IMPROVED
which medicines should be added to metformin 0.00 -> 0.00      30 -> 9       IMPROVED
how long does a lifestyle change program las 0.00 -> 0.00      52 -> 9       IMPROVED
how many sessions are in the first six month 0.00 -> 0.00      14 -> 67      REGRESSED
how much weight should someone lose to preve 0.00 -> 0.00    >100 -> 91      IMPROVED
how much exercise per week is recommended    1.00 -> 1.00       1 -> 1       same
what blood pressure target counts as optimal 0.00 -> 0.00      35 -> 5       IMPROVED
what signs suggest type 1 rather than type 2 0.00 -> 0.00      27 -> 6       IMPROVED
when should a newly diagnosed person be trea 0.00 -> 1.00       4 -> 1       IMPROVED

improved 7, regressed 1, same 2
mean precision@3: 0.13 -> 0.27


### Re-ranking

A cross-encoder reads the question and a chunk together and scores how well they match, which is slower however usually sharper than comparing embeddings. I'm using MedCPT, which was trained on PubMed search logs. It only re-sorts the top 30 results from the search, and everything after that keeps its order.

In [16]:
from sentence_transformers import CrossEncoder
WK_RERANK_MODEL = 'ncbi/MedCPT-Cross-Encoder'
wk_reranker = CrossEncoder(WK_RERANK_MODEL, max_length=512)
WK_RERANK_N = 30   # how many candidates the re-ranker re-sorts

def rerank(q, ranked, n=WK_RERANK_N):
    pool = ranked[:n]
    scores = wk_reranker.predict([(q, ch) for ch in pool])
    order = sorted(range(len(pool)), key=lambda i: scores[i], reverse=True)
    return [pool[i] for i in order] + ranked[n:]   # chunks past the pool keep their order

def search_rerank(q):
    return rerank(q, search_baseline(q))

def search_hyde_rerank(q):
    return rerank(q, search_hyde(q))   # HyDE finds candidates, original question re-ranks them

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### Measuring the re-ranker

Two runs, each compared to the baseline: the re-ranker on the plain search, and the re-ranker on top of HyDE. This is a slow cell, because it scores 30 chunks per question, especially on my laptop CPU.

In [17]:
wk_rerank = measure(search_rerank)
wk_hyde_rerank = measure(search_hyde_rerank)

print('Baseline vs baseline + re-rank\n')
compare(wk_baseline, wk_rerank)
print('\n\nBaseline vs HyDE + re-rank\n')
compare(wk_baseline, wk_hyde_rerank)

Baseline vs baseline + re-rank

question                                             P@3     first rank   verdict
what A1C level is used to diagnose diabetes  0.33 -> 0.33       1 -> 1       same
what fasting blood sugar confirms a diabetes 0.00 -> 0.00       5 -> 9       REGRESSED
which medicines should be added to metformin 0.00 -> 0.00      30 -> 24      IMPROVED
how long does a lifestyle change program las 0.00 -> 0.00      52 -> 52      same
how many sessions are in the first six month 0.00 -> 0.00      14 -> 21      REGRESSED
how much weight should someone lose to preve 0.00 -> 0.00    >100 -> >100    same
how much exercise per week is recommended    1.00 -> 1.00       1 -> 1       same
what blood pressure target counts as optimal 0.00 -> 0.00      35 -> 35      same
what signs suggest type 1 rather than type 2 0.00 -> 0.33      27 -> 2       IMPROVED
when should a newly diagnosed person be trea 0.00 -> 0.33       4 -> 1       IMPROVED

improved 3, regressed 2, same 5
mean precis

## Part 2: Measuring the effect

I ran the same 10 questions through five pipelines, all on the by-sentence chunks, and recorded precision@3 and the rank of the first chunk containing the answer. A question counts as improved if that chunk moved higher up the ranking, and regressed if it dropped lower.

| Pipeline | Mean P@3 | Improved / regressed / same vs baseline |
|---|---|---|
| Baseline | 0.13 | n/a |
| PRF | 0.17 | 3 / 4 / 3 |
| HyDE | 0.27 | 7 / 1 / 2 |
| Re-rank only | 0.20 | 3 / 2 / 5 |
| HyDE + re-rank | 0.20 | 6 / 2 / 2 |

One extra chunk moves the mean by about 0.03, so I put more weight on the rank column than on the means.

Which questions benefited: HyDE helped most where the plain search had the answer far down the list or missed it entirely. The first correct chunk moved up for metformin (30 to 9), the lifestyle program (52 to 9), blood pressure (35 to 5) and type 1 signs (27 to 6). The weight question moved from past 100 to 91, which is still too low to matter. The re-ranker only helped when the answer was already in the top 30 candidates: type 1 signs went from 27 to 2. It could not move anything outside that pool, which is why lifestyle, blood pressure and weight did not change.

Which did not: the A1C and exercise questions were already at rank 1, therefore nothing changed. PRF made 4 questions worse. For 8 of the 10 questions the baseline top 3 had no answer in it, therefore the words PRF added came from the wrong chunks (for the lifestyle program question it added "coaches training"). HyDE made the sessions question worse, which Part 3 looks at. The re-ranker pushed the fasting blood sugar answer down (5 to 9). Adding it on top of HyDE did worse than HyDE alone (0.27 to 0.20), and the ranks for fasting blood sugar, metformin, lifestyle and blood pressure all slipped. I think this could be because the re-ranker was trained on PubMed searches and my chunks are guideline text, but that would need to be explored and tested outide this assignment.

Five questions never reached the top 3 under any pipeline. The "newly diagnosed" jump to 1.00 under HyDE should be read carefully, because the fact I scored it on is just the word "insulin", which appears in over 100 chunks.

In [18]:
wk_all = {
    'baseline':    wk_baseline,
    'PRF':         wk_prf,
    'HyDE':        wk_hyde,
    'rerank':      wk_rerank,
    'HyDE+rerank': wk_hyde_rerank,
}
names = list(wk_all)
print(f"{'question':40s}" + ''.join(f'{n:>14s}' for n in names))
print(' ' * 40 + ''.join(f"{'P@3 / rank':>14s}" for _ in names))
for i, (q, _) in enumerate(qa):
    line = f'{q[:38]:40s}'
    for n in names:
        p, r = wk_all[n][i]
        line += f"{p:>6.2f} /{str(r) if r else '>100':>5s} "
    print(line)
print()
print(f"{'mean P@3':40s}" + ''.join(f'{sum(p for p, _ in wk_all[n]) / len(qa):>14.2f}' for n in names))

question                                      baseline           PRF          HyDE        rerank   HyDE+rerank
                                            P@3 / rank    P@3 / rank    P@3 / rank    P@3 / rank    P@3 / rank
what A1C level is used to diagnose dia    0.33 /    1   0.33 /    1   0.33 /    1   0.33 /    1   0.33 /    1 
what fasting blood sugar confirms a di    0.00 /    5   0.00 /    6   0.33 /    3   0.00 /    9   0.00 /    6 
which medicines should be added to met    0.00 /   30   0.00 /   29   0.00 /    9   0.00 /   24   0.00 /   13 
how long does a lifestyle change progr    0.00 /   52   0.00 /   77   0.00 /    9   0.00 /   52   0.00 /   12 
how many sessions are in the first six    0.00 /   14   0.00 /   21   0.00 /   67   0.00 /   21   0.00 /   67 
how much weight should someone lose to    0.00 / >100   0.00 / >100   0.00 /   91   0.00 / >100   0.00 /   91 
how much exercise per week is recommen    1.00 /    1   1.00 /    1   1.00 /    1   1.00 /    1   1.00 /    1 
w

## Part 3: A failure case

I picked the sessions question, where HyDE made the answer harder to find. The cell below prints the passage Gemini wrote, the top 3 chunks before and after, and where the first chunk containing "16 sessions" landed in each ranking.

In [19]:
q, fact = qa[4]
print('PASSAGE GEMINI WROTE (used for the search):')
print(hyde_text(q))

print('\nBASELINE top 3:'); peek(4, search_baseline)
print('\nHYDE top 3:');     peek(4, search_hyde)

for name, search in [('baseline', search_baseline), ('HyDE', search_hyde)]:
    ranked = search(q)
    pos = next((n for n, ch in enumerate(ranked, start=1) if fact.lower() in ch.lower()), None)
    if pos is None:
        print(f'\nFirst chunk containing "{fact}" under {name}: not in the top {WK_DEPTH}')
    else:
        print(f'\nFirst chunk containing "{fact}" under {name}, rank {pos}:')
        print(f'  ({SOURCES[WK_CONFIG][WK_CHUNKS.index(ranked[pos-1])]}) {ranked[pos-1][:300]}')

PASSAGE GEMINI WROTE (used for the search):
The core curriculum of the National Diabetes Prevention Program (DPP) requires the delivery of a minimum of 16 weekly sessions during the initial six-month intensive lifestyle intervention phase. These structured, evidence-based sessions focus on behavior modification, dietary counseling, and physical activity prescription to facilitate a target weight reduction of 5% to 7% for type 2 diabetes risk reduction. Following this initial six-month period, the program transitions to monthly maintenance sessions to support long-term adherence and sustained metabolic improvements.

BASELINE top 3:
how many sessions are in the first six months of the program   | fact: 16 sessions
  [miss    ] (dprp-standards)  By the beginning of Month 7, 7 of 10 eligible participants were still in the program (Retention=7/10=70%). • Month 7 retention was 70% (70% ≥ 40% re
  [miss    ] (dprp-standards) Eve ry 6 months, beginni ng at 12 or 1 8 months from the effective

The sessions question got worse with HyDE. The first chunk containing "16 sessions" dropped in rank, and neither version had a relevant chunk in the top 3.

Gemini's passage started with the right answer (16 weekly sessions), but I asked for 2 to 3 sentences, therefore it kept going with weight loss targets and monthly maintenance. The search used the whole passage, so it was pulled toward those topics instead of session counts.

HyDE assumes the made-up passage stays on the question and sounds like the document. Here it wandered, and the answer is a short requirements row that reads nothing like flowing prose.